<div class = "alert alert-block alert-info">
    <b>CCDR Inventory:</b> 306 variables
</div>

In [79]:
import os
import pandas as pd
import wbgapi as wb # https://github.com/tgherzog/wbgapi
import plotly.express as px
import plotly.graph_objects as go
import datetime
import numpy as np
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
# CCDR is number 87
wb.source.info()

id,name,lastupdated
1,Doing Business,2021-08-18
2,World Development Indicators,2021-09-15
3,Worldwide Governance Indicators,2021-09-27
5,Subnational Malnutrition Database,2016-03-21
6,International Debt Statistics,2021-01-21
11,Africa Development Indicators,2013-02-22
12,Education Statistics,2020-12-20
13,Enterprise Surveys,2021-04-02
14,Gender Statistics,2021-09-27
15,Global Economic Monitor,2020-07-27


In [4]:
# 306 variables
desc = pd.DataFrame(list(wb.series.list(db=87)))
desc = desc.rename({'id': 'variable', 'value': 'description'}, axis=1)
desc = desc.sort_values(by='variable', ascending=True)
desc[["code1", "code2", "code3", "code4", "code5"]] = desc['variable'].str.split(expand=True, pat='.')
desc.to_csv('desc.csv')
desc

,variable,description,code1,code2,code3,code4,code5
0,AG.LND.AGRI.ZS,Agricultural land (% of land area),AG,LND,AGRI,ZS,None
1,AG.LND.ARBL.ZS,Arable land (% of land area),AG,LND,ARBL,ZS,None
2,AG.LND.FRST.ZS,Share of surface occupied by forest (% of total),AG,LND,FRST,ZS,None
3,CC.ACLD.PRTS.NO,Total protests and riots by event type since 2018,CC,ACLD,PRTS,NO,None
4,CC.ADPO.MAEX.AA,Additional population exposed to annual coasta...,CC,ADPO,MAEX,AA,None
...,...,...,...,...,...,...,...
301,SI.POV.UMIC,Poverty headcount ratio at $5.50 a day (2011 P...,SI,POV,UMIC,None,None
302,SL.TLF.TOTL.FE.ZS,"Labor force, female (% of total labor force)",SL,TLF,TOTL,FE,ZS
303,SL.UEM.TOTL.ZS,"Unemployment, total (% of total labor force) (...",SL,UEM,TOTL,ZS,None
304,SP.POP.TOTL,"Population, total",SP,POP,TOTL,None,None


In [5]:
ccdr_306 = list(desc['variable'])
len(ccdr_306)

306

In [7]:
appended_data = []
for count, variable in enumerate(ccdr_306, start=1):
    print(count, variable)
    df = wb.data.DataFrame(series = variable, 
                           db = 87, 
                           labels = True).reset_index()
    df["variable"] = variable
    print("Done")
    appended_data.append(df)
    
appended_data = pd.concat(appended_data)

1 AG.LND.AGRI.ZS
Done
2 AG.LND.ARBL.ZS
Done
3 AG.LND.FRST.ZS
Done
4 CC.ACLD.PRTS.NO
Done
5 CC.ADPO.MAEX.AA
Done
6 CC.ADPO.MAEX.BB
Done
7 CC.ADPO.MIEX.AA
Done
8 CC.ADPO.MIEX.BB
Done
9 CC.AG.NTR.TOHA
Done
10 CC.ASP.COVG.QT
Done
11 CC.ASP.COVG.ZS
Done
12 CC.ASP.TRAF.PC
Done
13 CC.AVPB.PTPI.AI
Done
14 CC.AVPB.PTPI.AR
Done
15 CC.AVPB.PTPI.DI
Done
16 CC.AVPB.PTPI.FP
Done
17 CC.AVPB.PTPI.HE
Done
18 CC.AVPB.PTPI.LP
Done
19 CC.AVPB.TPOP.AG
Done
20 CC.AVPB.TPOP.AI
Done
21 CC.AVPB.TPOP.DI
Done
22 CC.AVPB.TPOP.HE
Done
23 CC.AVPB.TPOP.TE
Done
24 CC.CHIC.BTFP.AG
Done
25 CC.CHIC.BTFP.AI
Done
26 CC.CHIC.BTFP.DI
Done
27 CC.CHIC.BTFP.HE
Done
28 CC.CHIC.BTFP.TE
Done
29 CC.CHIC.CFPI.AG
Done
30 CC.CHIC.CFPI.AI
Done
31 CC.CHIC.CFPI.DI
Done
32 CC.CHIC.CFPI.FP
Done
33 CC.CHIC.CFPI.HE
Done
34 CC.CHIC.CFPI.LP
Done
35 CC.CO2.EMSE.BF
Done
36 CC.CO2.EMSE.BL
Done
37 CC.CO2.EMSE.EH
Done
38 CC.CO2.EMSE.EL
Done
39 CC.CO2.EMSE.EN
Done
40 CC.CO2.EMSE.FE
Done
41 CC.CO2.EMSE.IL
Done
42 CC.CO2.EMSE.IP
Done
43 CC.CO2.EMSE.L

In [9]:
appended_data.to_csv('appended_data.csv')

In [114]:
# 220 countries
tableau = appended_data.groupby('variable').count().reset_index()
#tableau[["code1", "code2", "code3", "code4", "code5"]] = tableau['variable'].str.split(expand=True, pat='.')
tableau = tableau.merge(desc, left_on = 'variable', right_on = 'variable')
tableau["row"] = tableau["variable"] + ": " + tableau["description"]
tableau.set_index('variable', inplace = True)

In [115]:
counts = tableau[["code1", "code2", "code3", "code4", "code5"]].value_counts().to_frame('counts')
counts.sort_values(by=["code1", "code2", "code3", "code4", "code5"], ascending=True)

counts
code1 code2 code3 code4 code5        
CC    EG    CONS  COAL  PC          1
                  GAS   PC          1
                  OIL   PC          1
      EN    ATM   COAL  CE          1
      SE    ADT   LITR  ZS          1
GC    DOD   TOTL  GD    ZS          1
      TAX   TOTL  GD    ZS          1
NY    GDP   COAL  RT    ZS          1
            MINR  RT    ZS          1
            MKTP  KD    ZG          1
            NGAS  RT    ZS          1
            PETR  RT    ZS          1
            TOTL  RT    ZS          1
SL    TLF   TOTL  FE    ZS          1
SP    URB   TOTL  IN    ZS          1

In [116]:
#list(tableau.columns)
columns = [ 'YR1960',
 'YR1961',
 'YR1962',
 'YR1963',
 'YR1964',
 'YR1965',
 'YR1966',
 'YR1967',
 'YR1968',
 'YR1969',
 'YR1970',
 'YR1971',
 'YR1972',
 'YR1973',
 'YR1974',
 'YR1975',
 'YR1976',
 'YR1977',
 'YR1978',
 'YR1979',
 'YR1980',
 'YR1981',
 'YR1982',
 'YR1983',
 'YR1984',
 'YR1985',
 'YR1986',
 'YR1987',
 'YR1988',
 'YR1989',
 'YR1990',
 'YR1991',
 'YR1992',
 'YR1993',
 'YR1994',
 'YR1995',
 'YR1996',
 'YR1997',
 'YR1998',
 'YR1999',
 'YR2000',
 'YR2001',
 'YR2002',
 'YR2003',
 'YR2004',
 'YR2005',
 'YR2006',
 'YR2007',
 'YR2008',
 'YR2009',
 'YR2010',
 'YR2011',
 'YR2012',
 'YR2013',
 'YR2014',
 'YR2015',
 'YR2016',
 'YR2017',
 'YR2018',
 'YR2019',
 'YR2020',
 'YR2021',
 'YR2022',
 'YR2023',
 'YR2024',
 'YR2025',
 'YR2026',
 'YR2027',
 'YR2028',
 'YR2029',
 'YR2030',
 'YR2031',
 'YR2032',
 'YR2033',
 'YR2034',
 'YR2035',
 'YR2036',
 'YR2037',
 'YR2038',
 'YR2039',
 'YR2040',
 'YR2041',
 'YR2042',
 'YR2043',
 'YR2044',
 'YR2045',
 'YR2046',
 'YR2047',
 'YR2048',
 'YR2049',
 'YR2050',
 'YR2100',]

In [117]:
countZ = pd.DataFrame(tableau[columns].astype(bool).sum(axis=1), columns=['number_years'])
tableau = tableau.merge(countZ, left_on = 'variable', right_on = 'variable').sort_values(by='number_years', ascending=False)
tableau

,economy,Country,YR1960,YR1961,YR1962,YR1963,YR1964,YR1965,YR1966,YR1967,YR1968,YR1969,YR1970,YR1971,YR1972,YR1973,YR1974,YR1975,YR1976,YR1977,YR1978,YR1979,YR1980,YR1981,YR1982,YR1983,YR1984,YR1985,YR1986,YR1987,YR1988,YR1989,YR1990,YR1991,YR1992,YR1993,YR1994,YR1995,YR1996,YR1997,YR1998,YR1999,YR2000,YR2001,YR2002,YR2003,YR2004,YR2005,YR2006,YR2007,YR2008,YR2009,YR2010,YR2011,YR2012,YR2013,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024,YR2025,YR2026,YR2027,YR2028,YR2029,YR2030,YR2031,YR2032,YR2033,YR2034,YR2035,YR2036,YR2037,YR2038,YR2039,YR2040,YR2041,YR2042,YR2043,YR2044,YR2045,YR2046,YR2047,YR2048,YR2049,YR2050,YR2100,description,code1,code2,code3,code4,code5,row,number_years
variable,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
SP.URB.TOTL.IN.ZS,220,220,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,215,0,Urban population (% of total population),SP,URB,TOTL,IN,ZS,SP.URB.TOTL.IN.ZS: Urban population (% of tota...,91
SP.POP.TOTL,220,220,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,216,217,217,216,216,216,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,217,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,213,0,"Population, total",SP,POP,TOTL,None,None,"SP.POP.TOTL: Population, total",91
NY.GDP.PCAP.CD,220,220,99,100,103,103,103,112,114,117,119,119,125,127,127,127,128,130,131,134,133,134,145,148,149,150,151,153,155,159,162,162,176,171,174,177,181,191,191,191,193,195,200,200,205,205,205,205,206,206,205,205,206,208,207,208,208,207,206,206,206,199,176,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,GDP per capita (current US$),NY,GDP,PCAP,CD,None,NY.GDP.PCAP.CD: GDP per capita (current US$),61
NY.GDP.MKTP.CD,220,220,99,100,103,103,103,112,114,117,119,119,125,127,127,127,128,130,131,134,133,134,145,148,149,150,151,153,155,159,162,162,176,171,175,178,182,191,191,191,193,195,200,200,205,205,205,205,206,206,205,205,206,208,207,208,208,207,206,206,206,199,176,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,GDP (current US$),NY,GDP,MKTP,CD,None,NY.GDP.MKTP.CD: GDP (current US$),61
NY.GDP.MKTP.KD.ZG,220,220,0,93,96,96,96,96,102,105,106,108,105,116,116,116,116,118,120,122,128,128,129,141,145,148,148,151,152,156,159,161,162,171,172,175,176,178,188,188,191,192,193,197,198,203,203,204,204,205,203,204,204,204,204,204,206,205,204,204,203,198,179,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,GDP growth (annual %),NY,GDP,MKTP,KD,ZG,NY.GDP.MKTP.KD.ZG: GDP growth (annual %),60
AG.LND.ARBL.ZS,220,220,0,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,176,177,177,177,177,177,177,181,200,203,203,203,203,203,203,203,204,204,204,204,204,204,206,207,206,206,206,207,207,207,207,207,206,205,205,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Arable land (% of land area),AG,LND,ARBL,ZS,None,AG.LND.ARBL.ZS: Arable land (% of land area),58
AG.LND.AGRI.ZS,220,220,0,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,179,183,202,205,205,205,205,205,205,205,206,206,206,206,206,206,208,209,208,208,208,210,210,210,210,210,210,209,209,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Agricultural land (% of land area),AG,LND,AGRI,ZS,None,AG.LND.AGRI.ZS: Agricultural land (% of land a...,58
NY.GDP.TOTL.RT.ZS,220,220,0,0,0,0,0,0,0,0,0,0,128,129,129,129,130,132,133,136,135,132,146,149,150,151,152,154,156

In [118]:
countZ["number_years"].value_counts()

1     133
29     36
21     28
5      14
26     12
2      10
0       9
7       9
18      8
20      7
50      5
40      3
22      3
8       3
15      3
3       3
30      2
61      2
58      2
19      2
91      2
11      1
31      1
16      1
47      1
48      1
49      1
28      1
13      1
60      1
23      1
Name: number_years, dtype: int64

In [126]:
# Empty = 9
empty = countZ[countZ["number_years"]==0].reset_index()["variable"].to_list()

#["variable"].to_list()
len(empty)
empty

['AG.LND.FRST.ZS',
 'CC.EG.COAL.EMIS',
 'CC.EMP.MULT.NO',
 'CC.FLD.CITY.ZS',
 'CC.SE.COMP.ZS',
 'CC.TCFD.COMP.NO',
 'IC.FRM.FEMO.ZS',
 'SG.GEN.PARL.ZS',
 'SL.TLF.TOTL.FE.ZS']

In [124]:
# Cross-section = 133
crossSection = countZ[countZ["number_years"]==1].reset_index()["variable"].to_list()

In [103]:
# Time series = 164
timeSeries = countZ[countZ["number_years"]>1]["variable"].to_list()
len(timeSeries)

164

In [121]:
pd.options.display.float_format = '{:,.1f}'.format
percentages = tableau[columns]/220*100
data = percentages.values.round(1)
data

array([[97.7, 97.7, 97.7, ..., 97.7, 97.7,  0. ],
       [98.2, 98.2, 98.2, ..., 96.8, 96.8,  0. ],
       [45. , 45.5, 46.8, ...,  0. ,  0. ,  0. ],
       ...,
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ],
       [ 0. ,  0. ,  0. , ...,  0. ,  0. ,  0. ]])

In [122]:
fig = px.imshow(data,
                labels=dict(x = "Year", y = "Indicator", color = "Available"),
                x = columns,
                y = tableau["row"].str.slice(0,100).tolist(), 
                #width=1500, 
                height=5000,
                #color_continuous_scale = "portland_r"
               )
fig.update_layout(
    title="CCDR Variables: Availability and Number of Observations",
    coloraxis_colorbar=dict(
    thicknessmode="pixels", thickness=70,
    lenmode="pixels", len=400,
    yanchor="top", y=.95,
    ticks="outside", ticksuffix=" %",
    dtick=5)
)
fig.update_xaxes(side="top")
fig.write_html("../CCDRinventory.html")
#fig.write_json("../WDIinventory.json", pretty=True)